# Metric Alignment Gate Flip Audit v1

This diagnostic follows `short_volshock_bc_initialization_official_replay_v1`: the candidate passed ordinary fold backtest gates but failed official warm-start replay gates. The purpose is to classify whether the optimistic source gate came from relative-return overstatement, MaxDD understatement, or both, then route the next experiment away from blind PPO.

External basis:
- Qlib's backtest/report workflow treats benchmark and risk analysis as replay artifacts, so promotion-style selection should consume official replay metrics rather than ad hoc fold slices.
- Backtest-overfitting literature such as Bailey et al. on the probability of backtest overfitting motivates blocking source-only winners after a replay gate flip.
- FinRL and walk-forward RL practice motivate evaluating held-out periods through a consistent trading simulator before retaining policies.

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.path_diagnostics import find_worst_drawdown_intervals
from src.soxl_quant.core.metric_reconciliation import (
    audit_source_vs_official_metrics,
    summarize_source_vs_official_audit,
)

EXPERIMENT = "metric_alignment_gate_flip_audit_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/short_volshock_bc_initialization_official_replay_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
EXTERNAL_SOURCES = [
    "https://qlib.readthedocs.io/en/latest/component/report.html",
    "https://github.com/microsoft/qlib",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253",
    "https://arxiv.org/abs/2009.11189",
    "https://arxiv.org/abs/2011.09607",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if hasattr(value, "item"):
        return json_safe(value.item())
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

source_selection = load_json(SOURCE_CARD / "selection.json")
source_summary = load_json(SOURCE_CARD / "summary.json")
source_pk = pd.read_csv(SOURCE_CARD / "source_vs_official_pk.csv")
print({
    "experiment": EXPERIMENT,
    "source_experiment": source_selection.get("experiment"),
    "source_decision": source_selection.get("decision"),
    "pk_rows": int(len(source_pk)),
})
display(source_pk)


{'experiment': 'metric_alignment_gate_flip_audit_v1', 'source_experiment': 'short_volshock_bc_initialization_official_replay_v1', 'source_decision': 'short_volshock_bc_initialization_official_replay_gate_flip_no_promotion', 'pk_rows': 2}


,candidate,fold,source_bc_gate,official_gate,gate_flip,source_bc_relative_return,official_warm_relative_return,relative_delta_official_minus_source,source_bc_max_dd,official_warm_max_dd,maxdd_delta_official_minus_source,source_bc_avg_pos,official_warm_avg_pos,trades,slice_gate,daily_path
0,bc_initialization,wf_2022,True,False,True,66.727389,67.624396,0.897006,14.997907,22.261989,7.264082,0.119868,0.097005,42,True,runtime/model_selection/card/short_volshock_bc...
1,bc_initialization,wf_2023_2024,True,False,True,31.124073,5.018899,-26.105174,13.361110,16.693087,3.331977,0.138743,0.183114,49,True,runtime/model_selection/card/short_volshock_bc...


In [2]:
audit = audit_source_vs_official_metrics(
    source_pk,
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
summary = summarize_source_vs_official_audit(audit)
audit.to_csv(CARD / "metric_alignment_audit.csv", index=False)
write_json(CARD / "metric_alignment_summary.json", summary)

display(audit)
display(pd.DataFrame([summary]))


,candidate,fold,source_relative_return,official_relative_return,source_max_dd,official_max_dd,source_gate_pass,official_gate_pass,source_optimistic_gate_flip,relative_delta_official_minus_source,relative_overstatement_pp,maxdd_delta_official_minus_source,maxdd_understatement_pp,material_relative_overstatement,material_maxdd_understatement,primary_mismatch,required_action
0,bc_initialization,wf_2023_2024,31.124073,5.018899,13.361110,16.693087,True,False,True,-26.105174,26.105174,3.331977,3.331977,True,True,relative_and_drawdown_gate_flip,block_training_until_official_replay_metric_al...
1,bc_initialization,wf_2022,66.727389,67.624396,14.997907,22.261989,True,False,True,0.897006,0.000000,7.264082,7.264082,False,True,max_drawdown_understated_gate_flip,block_training_until_official_replay_metric_al...


,decision,rows,source_optimistic_gate_flip_count,official_gate_pass_count,max_relative_overstatement_pp,max_maxdd_understatement_pp,worst_candidate,worst_fold,worst_primary_mismatch,next_mechanism
0,source_backtest_optimistic_gate_flip_no_training,2,2,0,26.105174,7.264082,bc_initialization,wf_2023_2024,relative_and_drawdown_gate_flip,replace_training_selection_gate_with_official_...


## Bad Interval Screen

For every official replay daily path referenced by the PK table, rank the worst drawdown intervals. These are the windows to use for feature-difference or factor-state analysis; the source-only fold metrics are no longer allowed to authorize training.

In [3]:
interval_frames: list[pd.DataFrame] = []
for row in source_pk.to_dict("records"):
    daily_path = ROOT / str(row["daily_path"])
    daily = pd.read_csv(daily_path, parse_dates=["datetime"])
    # path_diagnostics accepts account/nav. Official replay daily paths store rebased strategy_nav.
    diagnostic_daily = daily.rename(columns={"strategy_nav": "account"}).copy()
    intervals = find_worst_drawdown_intervals(diagnostic_daily, top_n=3, min_depth=0.03)
    if intervals.empty:
        continue
    intervals.insert(0, "candidate", row["candidate"])
    intervals.insert(1, "fold", row["fold"])
    intervals.insert(2, "daily_path", str(daily_path.relative_to(ROOT)))
    action_col = daily.get("action", pd.Series("HOLD", index=daily.index)).astype(str).str.upper()
    trade_rows = []
    for item in intervals.to_dict("records"):
        start = pd.Timestamp(item["peak_date"])
        end = pd.Timestamp(item["trough_date"])
        mask = (daily["datetime"] >= start) & (daily["datetime"] <= end)
        trade_rows.append(int(action_col[mask].isin(["BUY", "SELL"]).sum()))
    intervals["interval_trades"] = trade_rows
    interval_frames.append(intervals)

bad_intervals = pd.concat(interval_frames, ignore_index=True) if interval_frames else pd.DataFrame()
bad_intervals.to_csv(CARD / "worst_official_drawdown_intervals.csv", index=False)
display(bad_intervals)


,candidate,fold,daily_path,rank,peak_date,trough_date,start,end,rows,max_dd,max_dd_fraction,return,avg_pos,peak_nav,trough_nav,interval_trades
0,bc_initialization,wf_2022,runtime/model_selection/card/short_volshock_bc...,1,2022-01-03,2022-11-03,2022-01-03,2022-11-03,212,22.2620,0.222620,-22.2620,0.086226,1.000000,0.777380,27
1,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,1,2024-03-07,2024-11-15,2024-03-07,2024-11-15,177,16.6931,0.166931,-16.6931,0.140472,1.250300,1.041586,28
2,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,2,2024-01-25,2024-01-31,2024-01-25,2024-01-31,5,5.9209,0.059209,-5.9209,0.421099,1.078817,1.014941,1
3,bc_initialization,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,3,2024-02-09,2024-02-20,2024-02-09,2024-02-20,7,4.8551,0.048551,-4.8551,0.482089,1.137090,1.081884,1


In [4]:
source_optimistic_count = int(summary.get("source_optimistic_gate_flip_count", 0))
relative_fail_count = int(audit["primary_mismatch"].astype(str).str.contains("relative").sum()) if not audit.empty else 0
maxdd_fail_count = int(audit["primary_mismatch"].astype(str).str.contains("drawdown").sum()) if not audit.empty else 0
if source_optimistic_count:
    decision = "metric_alignment_gate_flip_blocks_training"
    next_mechanism = "replace source training-selection gates with official replay metrics before any new PPO branch"
elif int(summary.get("official_gate_pass_count", 0)) == len(audit):
    decision = "metric_alignment_official_gate_passed_no_promotion"
    next_mechanism = "extend to full fixed-split and walk-forward retention review"
else:
    decision = "metric_alignment_official_gate_failed_no_promotion"
    next_mechanism = "drop candidate or redesign target before training"

conclusion = []
for row in audit.sort_values(["fold", "candidate"]).to_dict("records"):
    conclusion.append(
        f"{row['fold']}: {row['candidate']} source_gate={bool(row['source_gate_pass'])}, "
        f"official_gate={bool(row['official_gate_pass'])}, mismatch={row['primary_mismatch']}, "
        f"relative_overstatement={row['relative_overstatement_pp']:.2f}pp, "
        f"maxdd_understatement={row['maxdd_understatement_pp']:.2f}pp."
    )
if not bad_intervals.empty:
    for row in bad_intervals[bad_intervals["rank"].eq(1)].sort_values(["fold", "candidate"]).to_dict("records"):
        conclusion.append(
            f"{row['fold']} worst official interval: {row['peak_date']} to {row['trough_date']}, "
            f"MaxDD={row['max_dd']:.2f}%, avg_pos={row['avg_pos']:.3f}, trades={int(row['interval_trades'])}."
        )
conclusion.append("Do not launch longer PPO from source-only gate evidence; official replay metrics must be the selection surface.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["model_selection", "metric_reconciliation", "bad_interval_screen"],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1",
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "source_experiment": source_selection.get("experiment"),
    "source_decision": source_selection.get("decision"),
    "source_next_mechanism": source_selection.get("next_mechanism"),
    "audit_summary": summary,
    "relative_failure_window_count": relative_fail_count,
    "maxdd_failure_window_count": maxdd_fail_count,
    "promotion_allowed": False,
    "actor_retention": "no actor retained; this card is metric governance only",
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "metric_alignment_audit.csv",
        "metric_alignment_summary.json",
        "worst_official_drawdown_intervals.csv",
        "selection.json",
        "summary.json",
    ],
}
summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "audit": audit.to_dict("records"),
    "audit_summary": summary,
    "worst_official_drawdown_intervals": bad_intervals.to_dict("records") if not bad_intervals.empty else [],
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary_payload)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "metric_alignment_gate_flip_audit_v1",
  "created_at": "2026-05-26T17:15:41.021774+00:00",
  "decision": "metric_alignment_gate_flip_blocks_training",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "metric_reconciliation",
    "bad_interval_screen"
  ],
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "short_volshock_bc_initialization_official_replay_v1",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "source_experiment": "short_volshock_bc_initialization_official_replay_v1",
  "source_decision": "short_volshock_bc_initialization_official_replay_gate_flip_no_promotion",
  "source_next_mechanism": "treat ordinary fold backtest as optimistic; repair metric alignment before PPO or actor retention",
  "audit_summary": {
    "decision": "source_backtest_optimistic_gate_flip_no_training",
    "rows": 2,
    "source_optimistic_gate_flip_count": 2,
    "official_gate_pass_count": 0,
  

## Conclusion

The source fold backtest is classified as optimistic because it passed both source gates while official replay failed both gates. The next optimization loop should use official replay metrics as the training-selection gate before PPO, then apply factor/feature-difference analysis to the worst official drawdown intervals.

In [5]:
summary_payload = load_json(CARD / "summary.json")
lines = ["# Runtime Conclusion", ""]
lines.append(f"- decision: `{summary_payload['decision']}`")
lines.append(f"- next: `{summary_payload['next_mechanism']}`")
lines.append("- promotion_allowed: `False`")
lines.append("")
for item in summary_payload.get("conclusion", []):
    lines.append(f"- {item}")
display(Markdown("\n".join(lines)))


# Runtime Conclusion

- decision: `metric_alignment_gate_flip_blocks_training`
- next: `replace source training-selection gates with official replay metrics before any new PPO branch`
- promotion_allowed: `False`

- wf_2022: bc_initialization source_gate=True, official_gate=False, mismatch=max_drawdown_understated_gate_flip, relative_overstatement=0.00pp, maxdd_understatement=7.26pp.
- wf_2023_2024: bc_initialization source_gate=True, official_gate=False, mismatch=relative_and_drawdown_gate_flip, relative_overstatement=26.11pp, maxdd_understatement=3.33pp.
- wf_2022 worst official interval: 2022-01-03 to 2022-11-03, MaxDD=22.26%, avg_pos=0.086, trades=27.
- wf_2023_2024 worst official interval: 2024-03-07 to 2024-11-15, MaxDD=16.69%, avg_pos=0.140, trades=28.
- Do not launch longer PPO from source-only gate evidence; official replay metrics must be the selection surface.
- Next mechanism: replace source training-selection gates with official replay metrics before any new PPO branch.